# 04 — 45-case synthetic ground-truth skeleton recovery

This benchmark now contains **exactly 45 admissible reconstruction tests**. It preserves the original 38 cases unchanged and adds seven genuinely distinct connected bridgeless cubic graph families: **Frucht, Möbius–Kantor, Desargues, Pappus, truncated tetrahedron, truncated cube, and Tutte**.

Each volume is Lee-skeletonized once and the identical skeleton is supplied to the historical `poly2graph` backend and the production topology-aware extractor. The optimized path must recover all 45 abstract graphs and must remain faster in median extraction time.

The truncated-tetrahedron challenge is particularly important: the first optimized selector returns an apparently clean all-trivalent 14-vertex/21-edge graph at zero radius, although the correct graph has 12 vertices and 18 edges. The new persistence selector is designed to detect this class of clean-looking split-junction failure without using ground truth.

Yamada validation remains separate and is evaluated only on embedded graphs with maximum degree $\le 2$.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Run inside the KnottedGraph checkout.')

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error

print('Python executable:', sys.executable)
print('KnottedGraph:', Path(knotted_graph.__file__).resolve())
print('Native Yamada backend:', native_available())
print('Native import error:', native_import_error())
assert native_available(), native_import_error()

In [ ]:
script = ROOT / 'dev' / 'run_skeletonization_45_validation.py'
proc = subprocess.run(
    [sys.executable, str(script)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError(
        f'45-case skeletonization validation failed.\n'
        f'STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}'
    )
assert 'TOTAL=45' in proc.stdout
assert 'OPTIMIZED=45/45' in proc.stdout
assert 'NEW_CHALLENGE_CASES=7/7' in proc.stdout
assert 'PASS: 45/45 optimized reconstructions' in proc.stdout

## Acceptance criterion

A pass requires all **45/45** optimized reconstructions, all **7/7** newly added challenge families, a strict median runtime improvement over `poly2graph` with at least a 1.5× extraction speedup, and successful degree-$\le2$ Yamada deformation checks.